In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Define sigmoid activation function and its derivative
def sigmoid(x):
    return 1/(1 + np.exp(-x))

def dsigmoid(x):
    sig = sigmoid(x)
    return sig * (1 - sig)

In [ ]:
# Data generation
## True values of parameters
w = np.array([1,1])
beta = np.array([2,1.5])
sig = 0.1

## data
n = 50
np.random.seed(42)
eps = np.random.normal(0,sig,n)
x = np.linspace(-10,20,n)

mu = beta[0] + beta[1]*sigmoid(w[0] + w[1]*x)

y = mu + eps

mu_dot = np.column_stack((beta[1]*dsigmoid(w[0] + w[1]*x), beta[1]*dsigmoid(w[0] + w[1]*x)*x, np.repeat(1,n), sigmoid(w[0] + w[1]*x)))
# mu_dot.shape, np.linalg.det(mu_dot.T @ mu_dot)

In [ ]:
# Define constants and functions for IF calculations
def C00(beta):
    return 1/(((1+beta)**0.5)*(2*np.pi)**(beta/2))

def C02(beta):
    return 1/(((1+beta)**1.5)*(2*np.pi)**(beta/2))

def C22(beta):
    return 3/(((1+beta)**2.5)*(2*np.pi)**(beta/2))

def psi_1(beta, x):
    return -(2*np.pi)**(-beta/2) * x * np.exp(-beta*x**2/2)

def psi_2(beta, x):
    return (1 - x**2) * (2*np.pi)**(-beta/2) * np.exp(-beta*x**2/2)

def mu_d(x):
    return np.array([beta[1]*dsigmoid(w[0] + w[1]*x), beta[1]*dsigmoid(w[0] + w[1]*x)*x, 1.0, sigmoid(w[0] + w[1]*x)])

In [ ]:
# IF calculations
def IF_i(t, i, alpha):
    return (1+alpha)**1.5 * (t - mu[i]) * np.exp(-alpha/(2*sig**2) * (t - mu[i])**2) * np.linalg.inv(mu_dot.T @ mu_dot) @ mu_dot[i,:]

def IF(t, alpha):
    A = np.array([IF_i(t,i,alpha=alpha) for i in range(n)])
    return A.sum(axis=0)


def IF_sig_i(t, i, beta):
    return -sig/(n*C22(beta)) * (psi_2(beta, (t-mu[i])/sig) - beta*C00(beta)/(1+beta))

def IF_sig(t, beta):
    a = np.array([IF_sig_i(t,i,beta) for i in range(n)])
    return a.sum()


def IF_pred_i(t, i, beta, x0):
    return -sig/(n*C02(beta)) * psi_1(beta, (t-mu[i])/sig) * (mu_d(x0).T @ np.linalg.inv(mu_dot.T @ mu_dot) @ mu_d(x0))

def IF_pred(t, beta, x0):
    a = np.array([IF_pred_i(t, i, beta, x0) for i in range(n)])
    return a.sum()

In [ ]:
cl = ['black', 'red','blue', 'green', 'orange', 'cyan']
linestyles = ['-', '--', '-.', ':', (0, (5, 1)), (0, (3, 5, 1, 5))]

In [ ]:
# Plotting IFs
def plot_IF(tt, i):
    M0  = np.array([IF_i(t, i, alpha=0) for t in tt])
    M1  = np.array([IF_i(t, i, alpha=0.1) for t in tt])
    M3  = np.array([IF_i(t, i, alpha=0.3) for t in tt])
    M5  = np.array([IF_i(t, i, alpha=0.5) for t in tt])
    M7  = np.array([IF_i(t, i, alpha=0.7) for t in tt])
    M10 = np.array([IF_i(t, i, alpha=1) for t in tt])
    M = np.array([M0, M1, M3, M5, M7, M10])
    return M

def plot_IF_sig(tt, i):
    M0  = np.array([IF_sig_i(t, i, beta=0) for t in tt])
    M1  = np.array([IF_sig_i(t, i, beta=0.1) for t in tt])
    M3  = np.array([IF_sig_i(t, i, beta=0.3) for t in tt])
    M5  = np.array([IF_sig_i(t, i, beta=0.5) for t in tt])
    M7  = np.array([IF_sig_i(t, i, beta=0.7) for t in tt])
    M10 = np.array([IF_sig_i(t, i, beta=1) for t in tt])
    M = np.array([M0, M1, M3, M5, M7, M10])
    return M

def plot_IF_pred(tt, i, x0):
    M0  = np.array([IF_pred_i(t, i, beta=0, x0=x0) for t in tt])
    M1  = np.array([IF_pred_i(t, i, beta=0.1, x0=x0) for t in tt])
    M3  = np.array([IF_pred_i(t, i, beta=0.3, x0=x0) for t in tt])
    M5  = np.array([IF_pred_i(t, i, beta=0.5, x0=x0) for t in tt])
    M7  = np.array([IF_pred_i(t, i, beta=0.7, x0=x0) for t in tt])
    M10 = np.array([IF_pred_i(t, i, beta=1, x0=x0) for t in tt])
    M = np.array([M0, M1, M3, M5, M7, M10])
    return M

In [ ]:
# IF Plot for data point index i = 2
tt = np.arange(1.3,2.7,0.01)
M = plot_IF(tt = tt, i=1)
print(f'IF Plot for data point index i = 2.')

tt_sig = np.arange(1,3,0.01)
M_sig = plot_IF_sig(tt = tt_sig, i=1)

tt_pred = np.arange(1.5, 2.5, 0.01)
M_pred = plot_IF_pred(tt_pred, i = 1, x0=4)

fig, axs = plt.subplots(3, 2, figsize=(14, 15))
lg = [r'$\beta = 0$', r'$\beta = 0.1$', r'$\beta = 0.3$', r'$\beta = 0.5$', r'$\beta = 0.7$', r'$\beta = 1$']
for j in range(6):
    axs[0,0].plot(tt, M[j][:,0], color = cl[j], linestyle = linestyles[j])
    axs[0,0].set_xlim(1.3,2.7); axs[0,0].set_ylim(-0.02,0.02)
    axs[0,0].set_title(r'IF of $w_0$', fontsize=20)
    #axs[0,0].legend(loc='best', labels=lg)
for j in range(6):
    axs[0,1].plot(tt, M[j][:,1], color = cl[j], linestyle = linestyles[j])
    axs[0,1].set_xlim(1.3,2.7); axs[0,1].set_ylim(-0.08,0.08)
    axs[0,1].set_title(r'IF of $w_1$', fontsize=20)
for j in range(6):
    axs[1,0].plot(tt, M[j][:,2], color = cl[j], linestyle = linestyles[j])
    axs[1,0].set_xlim(1.3,2.7); axs[1,0].set_ylim(-0.05,0.05)
    axs[1,0].set_title(r'IF of $w_0^{out}$', fontsize=20)
for j in range(6):
    axs[1,1].plot(tt, M[j][:,3], color = cl[j], linestyle = linestyles[j])
    axs[1,1].set_xlim(1.3,2.7); axs[1,1].set_ylim(-0.05,0.05)
    axs[1,1].set_title(r'IF of $w_1^{out}$', fontsize=20)
    
for j in range(6):
    axs[2,0].plot(tt_sig, M_sig[j], color = cl[j], linestyle = linestyles[j])
    axs[2,0].set_xlim(1,3)
    axs[2,0].set_ylim(-0.003, 0.01)
    axs[2,0].set_title(r'IF of $\sigma$', fontsize=20)
    
for j in range(6):
    axs[2,1].plot(tt_pred, M_pred[j], color = cl[j], linestyle = linestyles[j])
    axs[2,1].set_xlim(1.5,2.5)
    axs[2,1].set_ylim(-0.0003, 0.0003)
    axs[2,1].set_title(r'IF of $\mu_{n,\beta}^*(x)$ at $x=4$', fontsize=20)
    
for ax in axs.flat:
    ax.set_xlabel('t', fontsize=20); ax.set_ylabel('IF', fontsize=20)
    ax.tick_params(axis='both', labelsize=15)
    ax.legend(loc='best', labels=lg, fontsize=11)
    

plt.tight_layout()
plt.show()

In [ ]:
# IF Plot for data point index i = 49
tt = np.arange(2.9,4.1,0.01)
M = plot_IF(tt = tt, i = 48)
print(f'IF Plot for data point index i = 49.')

tt_sig = np.arange(2.5,4.5,0.01)
M_sig = plot_IF_sig(tt = tt_sig, i=49)

tt_pred = np.arange(3, 4, 0.01)
M_pred = plot_IF_pred(tt_pred, i=49, x0=4)

fig, axs = plt.subplots(3, 2, figsize=(14, 15))
lg = [r'$\beta = 0$', r'$\beta = 0.1$', r'$\beta = 0.3$', r'$\beta = 0.5$', r'$\beta = 0.7$', r'$\beta = 1$']
for j in range(6):
    axs[0,0].plot(tt, M[j][:,0], color = cl[j], linestyle = linestyles[j])
    axs[0,0].set_xlim(2.9,4.1); axs[0,0].set_ylim(-0.06,0.06)
    axs[0,0].set_title(r'IF of $w_0$', fontsize=20)
    #axs[0,0].legend(loc='best', labels=lg)
for j in range(6):
    axs[0,1].plot(tt, M[j][:,1], color = cl[j], linestyle = linestyles[j])
    axs[0,1].set_xlim(2.9,4.1); axs[0,1].set_ylim(-0.03,0.03)
    axs[0,1].set_title(r'IF of $w_1$', fontsize=20)
for j in range(6):
    axs[1,0].plot(tt, M[j][:,2], color = cl[j], linestyle = linestyles[j])
    axs[1,0].set_xlim(2.9,4.1); axs[1,0].set_ylim(-0.002,0.002)
    axs[1,0].set_title(r'IF of $w_0^{out}$', fontsize=20)
for j in range(6):
    axs[1,1].plot(tt, M[j][:,3], color = cl[j], linestyle = linestyles[j])
    axs[1,1].set_xlim(2.9,4.1); axs[1,1].set_ylim(-0.02,0.02)
    axs[1,1].set_title(r'IF of $w_1^{out}$', fontsize=20)
    
for j in range(6):
    axs[2,0].plot(tt_sig, M_sig[j], color = cl[j], linestyle = linestyles[j])
    axs[2,0].set_xlim(2.8,4.2)
    axs[2,0].set_ylim(-0.002, 0.01)
    axs[2,0].set_title(r'IF of $\sigma$', fontsize=20)
    
for j in range(6):
    axs[2,1].plot(tt_pred, M_pred[j], color = cl[j], linestyle = linestyles[j])
    axs[2,1].set_xlim(3,4)
    axs[2,1].set_ylim(-0.0003, 0.0003)
    axs[2,1].set_title(r'IF of $\mu_{n,\beta}^*(x)$ at $x=4$', fontsize=20)
    
for ax in axs.flat:
    ax.set_xlabel('t', fontsize=20); ax.set_ylabel('IF', fontsize=20)
    ax.tick_params(axis='both', labelsize=15)
    ax.legend(loc='best', labels=lg, fontsize=11)
    

plt.tight_layout()
plt.show()